This notebook has been used to generate some first tests using GX. It is based on the GX getting started docs. It is not intended to be run externally but only for generating the json configuration which can then be executed from CRON or Airflow.

In [10]:
import great_expectations
context = great_expectations.get_context()
import logging

AttributeError: module 'great_expectations' has no attribute 'get_context'

In [1]:
import great_expectations as gx

In [ ]:
import yaml

In [ ]:
from datetime import date

In [ ]:
logging.basicConfig(level=logging.DEBUG, force = True)

In [ ]:
connection_string = """bigquery://world-fishing-827/tech_great_expectations_temp_ttl_7d?\
credentials_path=/mnt/encrypted_data/git/api_keys/world-fishing-827-02584bdf5326.json"""

In [ ]:
with open("great_expectations/datasources/datasources.yml", "r") as ymlfile:
    datasource_config = yaml.full_load(ymlfile)

In [ ]:
datasource_config.get("project")

'gfw-google-827'

In [ ]:
gx_project = datasource_config.get("project")
#we create a data source for each schema, e.g. pipe_ais_v3_alpha_published
#get datasource if it exists, otherwise create datasource
# WARNING: it's necessary to distinguish because running add_or_update_sql resets the datasource config
# TODO: create feature request to simply get datasource if it already exists
if gx_project in [ds.get("name") for ds in context.list_datasources()]:
    gx_datasource = context.get_datasource(gx_project)
else:
    gx_datasource = context.sources.add_or_update_sql(
        name=gx_project, connection_string=connection_string, create_temp_table=True
    )

DEBUG:great_expectations.datasource.fluent.fluent_base_model:SQLDatasource.__fields_set__ assets added
DEBUG:great_expectations.datasource.fluent.fluent_base_model:QueryAsset.__fields_set__ assets discarded
DEBUG:great_expectations.datasource.fluent.fluent_base_model:SplitterYearAndMonthAndDay.__fields_set__ assets discarded
DEBUG:great_expectations.datasource.fluent.fluent_base_model:QueryAsset.__fields_set__ assets discarded
DEBUG:great_expectations.datasource.fluent.fluent_base_model:SplitterYearAndMonthAndDay.__fields_set__ assets discarded
DEBUG:great_expectations.datasource.fluent.fluent_base_model:QueryAsset.__fields_set__ assets discarded
DEBUG:great_expectations.datasource.fluent.fluent_base_model:SplitterColumnValue.__fields_set__ assets discarded
DEBUG:great_expectations.datasource.fluent.fluent_base_model:SQLDatasource.__fields_set__ assets added
DEBUG:great_expectations.datasource.fluent.fluent_base_model:QueryAsset.__fields_set__ assets discarded
DEBUG:great_expectations.

In [ ]:
gx_datasource.get_asset_names()

{'messages',
 'satellite_timing_offsets',
 'segs_activity',
 'segs_activity_daily',
 'ssvids_identities',
 'ssvids_identities_daily',
 'stats_daily',
 'vessel_info'}

In [ ]:
context.list_expectation_suite_names()

['gfw-google-827.alerts.segs_activity_daily',
 'gfw-google-827.constraints.segs_activity_daily',
 'pipe_ais_v3_alpha_published.satellite_timing_offsets',
 'pipe_ais_v3_alpha_published.segs_activity_daily',
 'pipe_ais_v3_alpha_published.vessel_info']

In [ ]:
selected_expectation_suit_name = 'gfw-google-827.constraints.segs_activity_daily'
selected_expectation_suit = context.get_expectation_suite(selected_expectation_suit_name)

In [ ]:
segs_activity_daily_asset = gx_datasource.get_asset("segs_activity_daily")

In [ ]:
segs_activity_daily_br = segs_activity_daily_asset.build_batch_request({'date': date.fromisoformat('2023-01-01')})

In [ ]:
segs_activity_daily_batches = gx_datasource.get_batch_list_from_batch_request(segs_activity_daily_br)

DEBUG:great_expectations.datasource.fluent.fluent_base_model:SQLDatasource.__fields_set__ assets added
INFO:great_expectations.datasource.fluent.fluent_base_model:SQLDatasource.dict() - substituting config values
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing-827/jobs?prettyPrint=false HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/queries/a530ef41-511a-46d6-a918-b90ec7b8fa23?maxResults=0&location=US&prettyPrint=false HTTP/1.1" 200 None
INFO:great_expectations.datasource.data_connector.batch_filter:batch_slice: None was parsed to: slice(0, None, None)


In [15]:
segs_activity_daily_validator = context.get_validator_using_batch_list(selected_expectation_suit, segs_activity_daily_batches)

In [16]:
segs_activity_daily_validator.expect_column_values_to_be_unique('seg_id')
segs_activity_daily_validator.expect_column_values_to_not_be_null('seg_id')

hours_cols = [hours_col for hours_col in segs_activity_daily_validator.columns() if 'hour' in hours_col]
for current_hours_col in hours_cols:
    segs_activity_daily_validator.expect_column_values_to_be_between(
        column=current_hours_col, 
        min_value=0, 
        max_value=48, 
        strict_max=True
    )
    if current_hours_col != 'hours':
        segs_activity_daily_validator.expect_column_pair_values_a_to_be_greater_than_b(
            'hours', current_hours_col, or_equal=True
        )


  warnings.warn(

DEBUG:great_expectations.core.expectation_configuration:evaluation_parameters have already been built on this expectation


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/datasets/tech_great_expectations_temp_ttl_7d/tables/gx_temp_87be59e3?prettyPrint=false HTTP/1.1" 200 None
  sqlalchemy.util.warn(

  sqlalchemy.util.warn(

DEBUG:great_expectations.execution_engine.sqlalchemy_execution_engine:Attempting query SELECT count(*) AS "table.row_count" 
FROM gx_temp_87be59e3
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing-827/jobs?prettyPrint=false HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/queries/74d1b9d0-bac1-40f9-8726-34a28f149721?maxResults=0&location=US&prettyPrint=false HTTP/1.1" 200 None
DEBUG:great_expectations.execution_engine.sqlalchemy_execution_engine:SqlAlchemyExecutionEngine computed 1 metrics on domain_id 95f83f9bbb0d78dd2ac96113b54f62b4
DEBUG:urllib3.connectionpool:https://bigquery.googleapis

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/datasets/tech_great_expectations_temp_ttl_7d/tables/gx_temp_87be59e3?prettyPrint=false HTTP/1.1" 200 None
DEBUG:great_expectations.execution_engine.sqlalchemy_execution_engine:Attempting query SELECT count(*) AS "table.row_count" 
FROM gx_temp_87be59e3
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing-827/jobs?prettyPrint=false HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/queries/c9e2715a-f38c-4f34-931b-9f1bd9f0d2ca?maxResults=0&location=US&prettyPrint=false HTTP/1.1" 200 None
DEBUG:great_expectations.execution_engine.sqlalchemy_execution_engine:SqlAlchemyExecutionEngine computed 1 metrics on domain_id 95f83f9bbb0d78dd2ac96113b54f62b4
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing

...

DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/datasets/tech_great_expectations_temp_ttl_7d/tables/gx_temp_87be59e3?prettyPrint=false HTTP/1.1" 200 None
DEBUG:great_expectations.execution_engine.sqlalchemy_execution_engine:Attempting query SELECT count(*) AS "table.row_count" 
FROM gx_temp_87be59e3
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing-827/jobs?prettyPrint=false HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/queries/279659ea-85df-4f21-9b34-a6f3bf12ea7b?maxResults=0&location=US&prettyPrint=false HTTP/1.1" 200 None
DEBUG:great_expectations.execution_engine.sqlalchemy_execution_engine:SqlAlchemyExecutionEngine computed 1 metrics on domain_id 95f83f9bbb0d78dd2ac96113b54f62b4
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing

Calculating Metrics:   0%|          | 0/11 [00:00<?, ?it/s]

DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/datasets/tech_great_expectations_temp_ttl_7d/tables/gx_temp_87be59e3?prettyPrint=false HTTP/1.1" 200 None
DEBUG:great_expectations.execution_engine.sqlalchemy_execution_engine:Attempting query SELECT count(*) AS "table.row_count" 
FROM gx_temp_87be59e3
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing-827/jobs?prettyPrint=false HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/queries/dd493d50-bdb2-4672-8808-485393965c32?maxResults=0&location=US&prettyPrint=false HTTP/1.1" 200 None
DEBUG:great_expectations.execution_engine.sqlalchemy_execution_engine:SqlAlchemyExecutionEngine computed 1 metrics on domain_id 95f83f9bbb0d78dd2ac96113b54f62b4
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/datasets/tech_great_expectations_temp_ttl_7d/tables/gx_temp_87be59e3?prettyPrint=false HTTP/1.1" 200 None
DEBUG:great_expectations.execution_engine.sqlalchemy_execution_engine:Attempting query SELECT count(*) AS "table.row_count" 
FROM gx_temp_87be59e3
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing-827/jobs?prettyPrint=false HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/queries/52feb404-6ccc-4ff9-934f-7dc801022ad9?maxResults=0&location=US&prettyPrint=false HTTP/1.1" 200 None
DEBUG:great_expectations.execution_engine.sqlalchemy_execution_engine:SqlAlchemyExecutionEngine computed 1 metrics on domain_id 95f83f9bbb0d78dd2ac96113b54f62b4
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing

In [17]:
segs_activity_daily_validator.save_expectation_suite(discard_failed_expectations=False)

INFO:great_expectations.validator.validator:	11 expectation(s) included in expectation_suite.
